---

# Summary: The Complete Workflow

## Before (Manual Process)
```
1. Stop app manually
2. Open ollama_config.json
3. Find the right field to edit
4. Type the path (with correct escaping)
5. Save file
6. Restart app
7. Hope it worked...
```

## After (Easy Process) ✨
```
1. Click Settings ⚙️
2. Find "Ollama Configuration" 
3. Paste or browse to model path
4. Click "Save & Restart Ollama"
5. App shows real-time status
6. Done! ✓
```

---

## Quick Start Examples

### I have models on E: drive
```
Settings → Ollama Configuration
Model Path: E:\MyModels
Click: Save & Restart Ollama
```

### My Ollama is on another computer
```
Settings → Ollama Configuration
Ollama Host: http://192.168.1.100:11434
Click: Save & Restart Ollama
```

### I'm having startup issues
```
Settings → Ollama Configuration
Max Retries: 20 (increase from 10)
Retry Delay: 3 (increase from 2)
Click: Save & Restart Ollama
```

---

## Troubleshooting

| Problem | Solution |
|---------|----------|
| "Path does not exist" error | Use Windows file browser to navigate and confirm path |
| Ollama won't restart | Try manual PowerShell: `Start-Process ollama.exe` |
| Port already in use | Change port from 11434 to 11435 in settings |
| Settings don't apply | Check app logs for detailed error messages |

---

## Files Reference

| File | Purpose |
|------|---------|
| `ollama_config.json` | Main configuration (editable with any text editor) |
| `start-ollama.ps1` | PowerShell script that starts Ollama |
| `ollama_config_manager.py` | Backend module managing config loading/saving |
| `routes/ollama_config.py` | API endpoints for settings |
| `api.ts` | Frontend API client methods |

**That's it! No more manual Ollama startup needed.** 🎉

## User Feedback & Error Messages

The Settings panel provides clear feedback:

### Success Scenario
```
✓ Settings saved! Ollama is restarting...
[Status updates in real-time]
✓ Ollama is ready! New settings applied.
```

### Error Scenario - Invalid Path
```
❌ Configuration validation failed:
  - Model path does not exist: X:\NonExistent
[User sees error, can fix and retry]
```

### Error Scenario - Ollama Won't Start
```
⚠️ Settings saved, but Ollama failed to restart
Tried for 30 seconds with no response.

Possible fixes:
1. Check that model path is correct
2. Verify ollama.exe exists at that location
3. Ensure no other app is using port 11434
4. Check Windows Event Viewer for errors

[Retry] [Manual Restart] [View Logs]
```

### Success with Warning
```
✓ Settings saved! Ollama is restarting...
⚠️ Port 11434 was in use, trying port 11435...
✓ Ollama is ready on port 11435!
```

---

# Section 6: Validate and Save Configuration Updates

## Validation Logic

Before saving, the app validates that your settings make sense:

```python
def validate_ollama_settings(config):
    """
    Validate configuration before saving
    """
    errors = []
    
    # Validate model_path if provided
    if config.get('model_path'):
        path = Path(config['model_path']).expanduser().resolve()
        if not path.exists():
            errors.append(f"Model path does not exist: {config['model_path']}")
        if not path.is_dir():
            errors.append(f"Model path is not a folder: {config['model_path']}")
    
    # Validate ollama_executable if provided
    if config.get('ollama_executable'):
        exe_path = Path(config['ollama_executable']).expanduser().resolve()
        if not exe_path.exists():
            errors.append(f"Ollama executable not found: {config['ollama_executable']}")
        if not exe_path.suffix.lower() == '.exe':
            errors.append(f"Ollama executable must be .exe file")
    
    # Validate ollama_host format
    if config.get('ollama_host'):
        if not config['ollama_host'].startswith('http://') and not config['ollama_host'].startswith('https://'):
            errors.append("Ollama host must start with http:// or https://")
    
    # Validate port number
    if config.get('ollama_port'):
        if not (1 <= config['ollama_port'] <= 65535):
            errors.append("Port must be between 1 and 65535")
    
    # Validate retry settings
    if config.get('max_retries'):
        if config['max_retries'] < 1:
            errors.append("Max retries must be at least 1")
    
    if config.get('retry_delay_seconds'):
        if config['retry_delay_seconds'] < 1:
            errors.append("Retry delay must be at least 1 second")
    
    return errors

# Save configuration with validation
def save_ollama_config(new_config, config_file="ollama_config.json"):
    """
    Save configuration with validation
    """
    # Validate
    errors = validate_ollama_settings(new_config)
    
    if errors:
        print("❌ Configuration validation failed:")
        for error in errors:
            print(f"  - {error}")
        return False
    
    try:
        # Make backup of old config
        config_path = Path(config_file)
        if config_path.exists():
            backup_path = config_path.with_suffix('.json.backup')
            config_path.rename(backup_path)
            print(f"Backup created: {backup_path}")
        
        # Save new config
        with open(config_file, 'w') as f:
            json.dump(new_config, f, indent=2)
        
        print(f"✓ Configuration saved to {config_file}")
        return True
        
    except Exception as e:
        print(f"❌ Error saving config: {e}")
        return False
```

### Frontend Auto-Restart with Status Polling

```typescript
// Poll for Ollama status after restart
async function waitForOllamaReady(maxWaitSeconds = 30): Promise<boolean> {
  const startTime = Date.now();
  
  while (Date.now() - startTime < maxWaitSeconds * 1000) {
    try {
      const status = await api.getOllamaStatus();
      if (status.is_running) {
        return true;  // Ollama is ready!
      }
    } catch (error) {
      // Still waiting...
    }
    
    // Wait 1 second before checking again
    await new Promise(resolve => setTimeout(resolve, 1000));
  }
  
  return false;  // Timeout
}

// Usage in React component
const handleSaveAndRestart = async () => {
  try {
    setStatus('saving');
    
    // Save config
    await api.updateOllamaConfig(settings);
    
    setStatus('restarting');
    
    // Wait for Ollama to be ready
    const ready = await waitForOllamaReady(30);
    
    if (ready) {
      setStatus('success');
      setStatusMessage('✓ Ollama restarted successfully!');
      
      // Refresh status after 3 seconds
      setTimeout(() => setStatus('idle'), 3000);
    } else {
      setStatus('error');
      setStatusMessage('✗ Ollama did not restart. Check logs.');
    }
  } catch (error) {
    setStatus('error');
    setStatusMessage(`Error: ${error.message}`);
  }
};
```

---

# Section 5: Auto-Restart Trigger on Settings Change

## How Auto-Restart Works

When you click **"Save & Restart Ollama"**, the app:

1. **Validates** your new settings
2. **Saves** to `ollama_config.json`
3. **Calls** `POST /ollama/restart` endpoint
4. **Backend starts** the PowerShell startup script
5. **Script finds** ollama.exe with new path
6. **Ollama restarts** with new settings
7. **App detects** when Ollama is ready
8. **Shows success** message

### Backend Auto-Restart Implementation

```python
# routes/ollama_config.py

@router.post("/restart")
async def restart_ollama() -> Dict[str, Any]:
    """
    Restart Ollama server with current config
    """
    manager = get_manager()
    
    # Stop any existing Ollama processes (optional, handled by script)
    
    # Trigger startup script
    manager.start_ollama_async()
    
    return {
        "status": "restart_initiated",
        "message": "Ollama is restarting with new configuration...",
        "timestamp": datetime.now().isoformat()
    }

@router.post("/config/update")
async def update_ollama_config(update: OllamaConfigUpdate) -> Dict[str, Any]:
    """
    Update config AND auto-restart if critical settings changed
    """
    manager = get_manager()
    original_config = manager.get_config()
    
    # Build update dict
    updates = {k: v for k, v in update.dict().items() if v is not None}
    
    # Save config
    success = manager.save_config(updates)
    
    if not success:
        raise HTTPException(status_code=500, detail="Failed to save config")
    
    # Check if critical settings changed
    critical_fields = {'model_path', 'ollama_executable', 'ollama_host', 'ollama_port'}
    changed = set(updates.keys()) & critical_fields
    
    auto_restart = False
    if changed:
        # Auto-restart if model path or host changed
        auto_restart = True
        manager.start_ollama_async()
    
    return {
        "status": "updated",
        "config": manager.get_config(),
        "auto_restarted": auto_restart,
        "message": "Config saved" + (" and Ollama is restarting..." if auto_restart else "")
    }
```

### React Component for Settings

The Settings panel in the app is built with React and looks like this conceptually:

```typescript
// OllamaSettingsPanel.tsx
interface OllamaSettings {
  ollama_enabled: boolean;
  model_path: string;
  ollama_executable: string;
  ollama_host: string;
  ollama_port: number;
  max_retries: number;
  retry_delay_seconds: number;
  auto_restart_on_failure: boolean;
}

export function OllamaSettingsPanel() {
  const [settings, setSettings] = useState<OllamaSettings | null>(null);
  const [isSaving, setIsSaving] = useState(false);
  const [saveStatus, setSaveStatus] = useState<'idle' | 'saving' | 'success' | 'error'>('idle');
  
  // Load settings on mount
  useEffect(() => {
    const loadSettings = async () => {
      const config = await api.getOllamaConfig();
      setSettings(config);
    };
    loadSettings();
  }, []);
  
  // Handle save & restart
  const handleSaveAndRestart = async () => {
    if (!settings) return;
    
    setIsSaving(true);
    setSaveStatus('saving');
    
    try {
      // Save config
      await api.updateOllamaConfig(settings);
      setSaveStatus('success');
      
      // Trigger restart
      setTimeout(async () => {
        await api.restartOllama();
      }, 500);
      
    } catch (error) {
      setSaveStatus('error');
      console.error('Failed to save settings:', error);
    } finally {
      setIsSaving(false);
    }
  };
  
  if (!settings) return <div>Loading...</div>;
  
  return (
    <div className="ollama-settings">
      <h3>Ollama Configuration</h3>
      
      {/* Enable/Disable Toggle */}
      <label>
        <input 
          type="checkbox" 
          checked={settings.ollama_enabled}
          onChange={(e) => setSettings({...settings, ollama_enabled: e.target.checked})}
        />
        Enable Ollama Auto-Start
      </label>
      
      {/* Model Path */}
      <div>
        <label>Model Path:</label>
        <input 
          type="text" 
          value={settings.model_path}
          onChange={(e) => setSettings({...settings, model_path: e.target.value})}
          placeholder="e.g., E:\MyModels"
        />
        <button onClick={() => openFolderBrowser('model_path')}>Browse...</button>
      </div>
      
      {/* Ollama Executable */}
      <div>
        <label>Ollama Executable:</label>
        <input 
          type="text" 
          value={settings.ollama_executable}
          onChange={(e) => setSettings({...settings, ollama_executable: e.target.value})}
          placeholder="e.g., E:\MyModels\ollama.exe (leave empty to auto-detect)"
        />
      </div>
      
      {/* Ollama Host */}
      <div>
        <label>Ollama Host:</label>
        <input 
          type="text" 
          value={settings.ollama_host}
          onChange={(e) => setSettings({...settings, ollama_host: e.target.value})}
          placeholder="http://127.0.0.1:11434"
        />
      </div>
      
      {/* Retry Settings */}
      <div>
        <label>Max Retries:</label>
        <input 
          type="number" 
          value={settings.max_retries}
          onChange={(e) => setSettings({...settings, max_retries: parseInt(e.target.value)})}
        />
      </div>
      
      <div>
        <label>Retry Delay (seconds):</label>
        <input 
          type="number" 
          value={settings.retry_delay_seconds}
          onChange={(e) => setSettings({...settings, retry_delay_seconds: parseInt(e.target.value)})}
        />
      </div>
      
      {/* Status */}
      {saveStatus === 'saving' && <p className="info">Saving and restarting...</p>}
      {saveStatus === 'success' && <p className="success">✓ Settings saved! Ollama is restarting...</p>}
      {saveStatus === 'error' && <p className="error">✗ Failed to save settings. Check your input.</p>}
      
      {/* Save Button */}
      <button 
        onClick={handleSaveAndRestart}
        disabled={isSaving}
        className="primary-btn"
      >
        {isSaving ? 'Saving...' : 'Save & Restart Ollama'}
      </button>
    </div>
  );
}
```

---

# Section 4: Settings UI Implementation

## The Easy Way: Settings Panel in the App

Instead of editing `ollama_config.json` manually, the app provides a Settings panel.

### What the Settings Panel Looks Like

**Location**: Settings ⚙️ → Ollama Configuration

```
┌─────────────────────────────────────────┐
│        Ollama Configuration             │
├─────────────────────────────────────────┤
│                                         │
│ Enable Ollama Auto-Start: [✓] ON        │
│                                         │
│ Model Path:                             │
│ [E:\MyModels                          ] │
│  Browse...                              │
│                                         │
│ Ollama Executable:                      │
│ [E:\MyModels\ollama.exe               ] │
│  Browse... (or leave empty)             │
│                                         │
│ Ollama Host:                            │
│ [http://127.0.0.1:11434               ] │
│                                         │
│ Max Retries:                            │
│ [10]                                    │
│                                         │
│ Retry Delay (seconds):                  │
│ [2]                                     │
│                                         │
│ Auto-Restart on Failure: [✓] ON         │
│                                         │
├─────────────────────────────────────────┤
│  [Cancel]  [Save & Restart Ollama]      │
└─────────────────────────────────────────┘
```

### How to Use It

1. **Update Model Path**: 
   - Click in the text field and type the path
   - Or click "Browse..." to select a folder
   
2. **Update Ollama Host**:
   - Change only if running Ollama on a different PC
   - Format: `http://IP_ADDRESS:PORT`

3. **Adjust Retry Settings** (optional):
   - Increase `Max Retries` if slow PC
   - Increase `Retry Delay` if network is unstable

4. **Save Changes**:
   - Click **Save & Restart Ollama**
   - App automatically restarts with new settings
   - Status shows when Ollama is ready

---

# Section 3: Load and Parse Configuration

## How the App Reads Your Config

```python
import json
from pathlib import Path

def load_ollama_config(config_path="ollama_config.json"):
    """
    Load and parse ollama_config.json with validation
    """
    try:
        if not Path(config_path).exists():
            print(f"⚠️ Config not found at {config_path}")
            return get_default_config()
        
        with open(config_path, 'r') as f:
            config = json.load(f)
        
        print(f"✓ Loaded config from {config_path}")
        return config
        
    except json.JSONDecodeError as e:
        print(f"❌ Invalid JSON in config: {e}")
        return get_default_config()
    except Exception as e:
        print(f"❌ Error reading config: {e}")
        return get_default_config()

def get_default_config():
    """
    Return default config if file is missing or invalid
    """
    return {
        "ollama_enabled": True,
        "model_path": "",
        "ollama_executable": "",
        "ollama_host": "http://127.0.0.1:11434",
        "ollama_port": 11434,
        "max_retries": 10,
        "retry_delay_seconds": 2,
        "auto_restart_on_failure": True,
        "keep_alive": "5m"
    }

def validate_config(config):
    """
    Validate configuration and handle missing fields
    """
    required_fields = {
        "ollama_enabled": True,
        "model_path": "",
        "ollama_host": "http://127.0.0.1:11434",
        "ollama_port": 11434,
        "max_retries": 10,
        "retry_delay_seconds": 2
    }
    
    # Fill in missing fields with defaults
    for field, default_value in required_fields.items():
        if field not in config:
            config[field] = default_value
    
    return config
```

---

# Section 2: What You NEED to Change

## The Main Thing: Tell the App Where Models Are

Your models folder contains `ollama.exe` and your model files. You need to tell the app where this folder is.

### Example 1: Models on External Drive (Recommended for Low Disk Space)

**Problem**: Your C: drive doesn't have space for models

**Solution**: 
```json
{
  "model_path": "E:\\MyModels",
  "ollama_executable": "E:\\MyModels\\ollama.exe"
}
```

**What to change**:
- Replace `E:\\MyModels` with your actual drive letter and folder name
- Use double backslashes (`\\`) or forward slashes (`/`)

### Example 2: Models on System Drive with Ollama Installed

**Problem**: Ollama is already installed on your computer

**Solution**:
```json
{
  "model_path": "C:\\Users\\YourName\\.ollama\\models",
  "ollama_executable": ""
}
```

Leave `ollama_executable` empty and the app will find Ollama automatically.

### Example 3: Portable Ollama (ollama.exe in Models Folder)

**Problem**: You have a folder with `ollama.exe` and all models together

**Solution**:
```json
{
  "model_path": "D:\\PortableOllama",
  "ollama_executable": "D:\\PortableOllama\\ollama.exe"
}
```

### Example 4: Network Server (Ollama on Different PC)

**Problem**: Ollama is running on another computer (192.168.1.100)

**Solution**:
```json
{
  "ollama_host": "http://192.168.1.100:11434"
}
```

## Default Configuration (Fresh Install)

```json
{
  "ollama_enabled": true,
  "model_path": "",
  "ollama_executable": "",
  "ollama_host": "http://127.0.0.1:11434",
  "ollama_port": 11434,
  "max_retries": 10,
  "retry_delay_seconds": 2,
  "auto_restart_on_failure": true,
  "keep_alive": "5m",
  "notes": "This file can be edited manually. Empty paths will auto-detect."
}
```

### Field Descriptions

| Field | Type | Default | What It Does |
|-------|------|---------|--------------|
| `ollama_enabled` | boolean | `true` | Enable/disable Ollama auto-start |
| `model_path` | string | `""` | Where your models are stored (MAIN THING TO CHANGE) |
| `ollama_executable` | string | `""` | Path to `ollama.exe` (usually auto-detected) |
| `ollama_host` | string | `http://127.0.0.1:11434` | Where Ollama server runs |
| `ollama_port` | integer | `11434` | Port number (change if 11434 is in use) |
| `max_retries` | integer | `10` | How many times to retry startup |
| `retry_delay_seconds` | integer | `2` | Wait time between retries |
| `auto_restart_on_failure` | boolean | `true` | Auto-restart if Ollama crashes |
| `keep_alive` | string | `5m` | Keep models in memory for 5 minutes |

---

# Section 1: Configuration File Structure

## File Location
```
python-backend/ollama_config.json
```

**Note**: You can also edit this file manually with any text editor (Notepad, VS Code, etc.) if needed

# Ollama Settings & Configuration Guide

## What You Need to Know

This notebook explains:
1. **Configuration File Structure** - What's in `ollama_config.json`
2. **What to Change** - Which fields to edit
3. **Settings UI** - Easy way to change settings from within the app (no file editing needed!)
4. **Auto-Restart** - App automatically restarts Ollama when you change critical settings

---

## Quick Reference: The "Backdoor" Approach

Instead of manually editing `ollama_config.json`, you'll use the **Settings Panel** in the app:

1. Click **Settings** (⚙️ icon in top-right)
2. Find **Ollama Configuration** section
3. Update model path, Ollama host, or other settings
4. Click **Save & Restart**
5. App automatically restarts Ollama with new settings

✨ **No file editing required!**